In [4]:
!pip install transformers datasets scikit-learn

import json
import re
import torch
import numpy as np
from datasets import Dataset
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

# =========================
# 1. Load Dataset
# =========================
with open("/content/ML_relations_fixed.json") as f:
    data = json.load(f)

# =========================
# 2. Extract Text + Labels
# =========================
texts = []
labels = []

for d in data:
    text = d["input"]
    label = d["output"].strip()
    texts.append(text)
    labels.append(label)

# Define label set
label_list = ["used_for", "based_on", "implements", "part_of", "improves", "no_relation"]

label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

# Handle unknown labels
labels = [l if l in label2id else "no_relation" for l in labels]

# =========================
# 3. Train-Test Split
# =========================
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

train_dataset = Dataset.from_dict({
    "text": train_texts,
    "label": [label2id[l] for l in train_labels]
})

val_dataset = Dataset.from_dict({
    "text": val_texts,
    "label": [label2id[l] for l in val_labels]
})

# =========================
# 4. Tokenization
# =========================
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=256)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# =========================
# 5. Model
# =========================
model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

# =========================
# 6. Metrics (Weighted Only)
# =========================
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision_weighted": precision,
        "recall_weighted": recall,
        "f1_weighted": f1
    }
# =========================
# 7. Training Arguments
# =========================
training_args = TrainingArguments(
    output_dir="./roberta-relation",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    learning_rate=2e-5,
    load_best_model_at_end=True,
   metric_for_best_model="f1_weighted"  # ✅ fixed key
)

# =========================
# 8. Trainer
# =========================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# =========================
# 9. Train
# =========================
trainer.train()

# =========================
# 10. FINAL EVALUATION + CLEAN PRINT (Weighted Only)
# =========================
metrics = trainer.evaluate()

print("\n" + "="*40)
print("📊 FINAL EVALUATION RESULTS")
print("="*40)

print(f"Accuracy            : {metrics['eval_accuracy']:.4f}")
print(f"Precision (Weighted): {metrics['eval_precision_weighted']:.4f}")
print(f"Recall (Weighted)   : {metrics['eval_recall_weighted']:.4f}")
print(f"F1 Score (Weighted) : {metrics['eval_f1_weighted']:.4f}")

print("="*40)
# =========================
# 11. OPTIONAL: CLASSIFICATION REPORT
# =========================
predictions = trainer.predict(val_dataset)
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

print("\n📋 Detailed Classification Report:\n")
print(classification_report(labels, preds, target_names=label_list, zero_division=0))

# =========================
# 12. Entity Extraction Function
# =========================
def extract_entities(text):
    e1 = re.search(r'Entity1:\s*(.*?)\s*\((.*?)\)', text)
    e2 = re.search(r'Entity2:\s*(.*?)\s*\((.*?)\)', text)

    entity1 = e1.group(1).strip() if e1 else ""
    entity1_type = e1.group(2).strip() if e1 else ""

    entity2 = e2.group(1).strip() if e2 else ""
    entity2_type = e2.group(2).strip() if e2 else ""

    return entity1, entity1_type, entity2, entity2_type

# =========================
# 13. Prediction + Triplets
# =========================
model.eval()

triplets = []

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

for text in val_texts:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        pred = torch.argmax(outputs.logits, dim=1).item()

    relation = id2label[pred]
    e1, e1_type, e2, e2_type = extract_entities(text)

    triplets.append({
        "entity1": e1,
        "entity1_type": e1_type,
        "relation": relation,
        "entity2": e2,
        "entity2_type": e2_type
        })

# =========================
# 14. Save Triplets
# =========================
with open("predicted_triplets.json", "w") as f:
    json.dump(triplets, f, indent=4)

print("\n✅ Triplets saved to predicted_triplets.json")

Map:   0%|          | 0/1569 [00:00<?, ? examples/s]

Map:   0%|          | 0/393 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,Accuracy,Precision Weighted,Recall Weighted,F1 Weighted
1,No log,1.034471,0.692112,0.674580,0.692112,0.674992
2,No log,0.918113,0.676845,0.654418,0.676845,0.659748
3,1.151829,0.831944,0.737913,0.763161,0.737913,0.726704
4,1.151829,0.752882,0.755725,0.767801,0.755725,0.742334
5,1.151829,0.737179,0.750636,0.761619,0.750636,0.737711


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


📊 FINAL EVALUATION RESULTS
Accuracy            : 0.7557
Precision (Weighted): 0.7678
Recall (Weighted)   : 0.7557
F1 Score (Weighted) : 0.7423

📋 Detailed Classification Report:

              precision    recall  f1-score   support

    used_for       0.83      0.86      0.84       118
    based_on       1.00      0.11      0.20        18
  implements       0.77      0.79      0.78        34
     part_of       0.75      0.58      0.65        57
    improves       0.71      0.80      0.75        49
 no_relation       0.70      0.81      0.75       117

    accuracy                           0.76       393
   macro avg       0.79      0.66      0.66       393
weighted avg       0.77      0.76      0.74       393


✅ Triplets saved to predicted_triplets.json
